In [0]:
%run ./01-config

In [0]:
import json
from pyspark.sql import functions as F

class Producer:
    def __init__(self):
        conf = Config()
        self.test_data_dir = conf.test_data_dir
        self.landing_zone = conf.landing_zone
        self.checkpoint_base = conf.checkpoint_base

        self.bootstrap_server = conf.BOOTSTRAP_SERVER
        self.kafka_username = conf.CLUSTER_API_KEY
        self.kafka_password = conf.CLUSTER_API_SECRET
        self.jaas_module = conf.JAAS_MODULE


    def user_registration(self, set_num):
        source = (f"{self.test_data_dir}/"f"1-registered_users_{set_num}.csv")
        target = (f"{self.landing_zone}/registered_users_bz/"f"1-registered_users_{set_num}.csv")
        dbutils.fs.cp(source, target)
        print(f"Registered users copied: {set_num}")


    def gym_logins(self, set_num):
        source = (f"{self.test_data_dir}/"f"5-gym_logins_{set_num}.csv")
        target = (f"{self.landing_zone}/gym_logins_bz/"f"5-gym_logins_{set_num}.csv")
        dbutils.fs.cp(source, target)
        print(f"Gym logins copied: {set_num}")


    def read_json(self, file_name):
        file_path = f"{self.test_data_dir}/{file_name}"
        schema = """
            key STRING,
            value STRING,
            topic STRING,
            partition INT,
            offset LONG,
            timestamp LONG
        """

        return (
                    spark.read.schema(schema).json(file_path)
                )


    def get_kafka_message(self, df):

        return (
                    df.select(
                        F.col("key").cast("string").alias("key"),
                        F.col("value").cast("string").alias("value"),
                        F.col("timestamp")
                    )
                    .withColumn(
                        "timestamp",
                        F.expr("timestamp_millis(timestamp * 1000)")
                    )
                )


    def send_to_kafka(self, kafka_df, topic):
        checkpoint_path = (f"{self.checkpoint_base}/"f"kafka_producer/{topic}")
        kafka_output = kafka_df.select("key", "value")

        (
            kafka_output.write
            .format("kafka")
            .option(
                "kafka.bootstrap.servers",
                self.bootstrap_server
            )
            .option("kafka.security.protocol", "SASL_SSL")
            .option("kafka.sasl.mechanism", "PLAIN")
            .option("kafka.sasl.jaas.config",f'{self.jaas_module} required 'f'username="{self.kafka_username}" 'f'password="{self.kafka_password}";')
            .option("topic", topic)
            .save()
        )

        print(f"Successfully published records to {topic}")


    def process_topic(self, file_name, topic):
        print(f"\nPublishing {file_name} to {topic}...")
        source_df = self.read_json(file_name)
        kafka_df = self.get_kafka_message(source_df)
        record_count = kafka_df.count()
        print(f"Input records: {record_count:,}")
        self.send_to_kafka(kafka_df, topic)
        print(f"Publishing completed: {topic}")


    def produce(self, set_num):
        if set_num not in [1, 2]:
            raise ValueError("Only dataset 1 and 2 are supported")

        print(f"\nProducing dataset {set_num}...")
        self.user_registration(set_num)
        self.gym_logins(set_num)
        self.process_topic(f"2-user_info_{set_num}.json","user_info")
        self.process_topic(f"4-workout_{set_num}.json","workout")
        self.process_topic(f"3-bpm_{set_num}.json","bpm")
        print(f"\nDataset {set_num} production completed.")
